# 作业 1：实现 Word2Vec 并在 PTB 数据集上训练

## 作业目标
在本次作业中，你将实现 **Skip-gram + Negative Sampling** 的 Word2Vec 模型，并在 **Penn Treebank (PTB)** 数据集上进行训练。  
通过补全给定的 Notebook 代码，你需要完成以下核心任务：负采样、批次构建、模型和损失定义，以及模型训练。

完成本作业后，你应该能够：
- 掌握 Word2Vec 的核心原理（Skip-gram + Negative Sampling）  
- 了解文本数据的预处理流程，包括构建词典、二次采样、生成训练样本  
- 使用 **PyTorch** 实现并训练一个简单的词向量模型  
- 观察训练得到的词向量效果，通过最近邻词测试验证词语语义相似性  

---

## 提交要求
1. 完整的 `.ipynb` 文件，包含你实现的 PyTorch 代码  
2. Notebook 的运行记录，这将包含每个epoch输出的loss记录
3. 词向量测试结果：给定几个 query token，输出它们的最近邻词，并写简要分析  

---

## Part 0: 导入依赖

In [1]:
import collections
import math
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import random
import sys
import time
import zipfile

## Part 1: 读取 PTB 数据并构建词典

PTB 数据集是语言模型常用的文本语料库。我们会对其进行以下处理：
- 读取数据
- 构建词典，过滤低频词（频率 < 5 的丢弃）
- 将单词映射为整数 ID

In [2]:
with open('data/ptb.train.txt', 'r') as f:
    lines = f.readlines()
    # st是sentence的缩写
    raw_dataset = [st.split() for st in lines]

In [3]:
# 为了计算简单，我们只保留在数据集中至少出现5次的词
counter = collections.Counter([tk for st in raw_dataset for tk in st])
counter = dict(filter(lambda x: x[1] >= 5, counter.items()))

# 然后将词映射到整数索引
idx_to_token = [tk for tk, _ in counter.items()]
token_to_idx = {tk: idx for idx, tk in enumerate(idx_to_token)}
dataset = [[token_to_idx[tk] for tk in st if tk in token_to_idx]
           for st in raw_dataset]
num_tokens = sum([len(st) for st in dataset])

## Part 2: 二次采样 (Subsampling)

高频词（如 "the", "a"）会主导训练，但它们并不携带太多语义信息。
我们对高频词进行二次采样，降低它们出现的概率。

In [4]:
def discard(idx):
    return random.uniform(0, 1) < 1 - math.sqrt(
        1e-4 / counter[idx_to_token[idx]] * num_tokens)

subsampled_dataset = [[tk for tk in st if not discard(tk)] for st in dataset]

## Part 3: 提取中心词和上下文词
我们将与中心词距离不超过背景窗口大小的词作为它的背景词。下面定义函数提取出所有中心词和它们的背景词。它每次在整数1和max_window_size（最大背景窗口）之间随机均匀采样一个整数作为背景窗口大小。

In [5]:
def get_centers_and_contexts(dataset, max_window_size):
    centers, contexts = [], []
    for st in dataset:
        if len(st) < 2:  # 每个句子至少要有2个词才可能组成一对“中心词-背景词”
            continue
        centers += st
        for center_i in range(len(st)):
            window_size = random.randint(1, max_window_size)
            indices = list(range(max(0, center_i - window_size),
                                 min(len(st), center_i + 1 + window_size)))
            indices.remove(center_i)  # 将中心词排除在背景词之外
            contexts.append([st[idx] for idx in indices])
    return centers, contexts

all_centers, all_contexts = get_centers_and_contexts(subsampled_dataset, 5)

In [6]:
all_centers[:3], all_contexts[:3]

([0, 3, 4], [[3], [0, 4, 6, 11], [3, 6]])

## Part 4: 负采样 (Negative Sampling)
我们使用负采样来进行近似训练。对于一对中心词和背景词，我们随机采样 K 个噪声词（实验中设 K=5）。根据word2vec论文的建议，噪声词采样概率 P(w) 设为 w 词频与总词频之比的0.75次方。

In [ ]:
def get_negatives(all_contexts, sampling_weights, K):
    all_negatives, neg_candidates, i = [], [], 0
    population = list(range(len(sampling_weights)))
    for contexts in all_contexts:
        negatives = []
        while len(negatives) < len(contexts) * K:
            #TODO:  1.当候选集用完时，重新采样 
            #       2.取出一个负例，并更新索引 
            #       3.如果负例不在背景词中，则加入 negatives
            if i == len(neg_candidates):
                neg_candidates = random.choices(population, sampling_weights, k=int(1e5))
                i = 0
            if neg_candidates[i] not in contexts:
                negatives.append(neg_candidates[i])
            i += 1

        all_negatives.append(negatives)
    return all_negatives

sampling_weights = [counter[w]**0.75 for w in idx_to_token]
all_negatives = get_negatives(all_contexts, sampling_weights, 5)


## Part 5: 构建批量 (Batchify)

为了高效训练，我们需要将训练样本打包成小批量，并对不同长度的上下文和负样本进行 **padding**：

- `centers`：中心词索引  
- `contexts_negatives`：上下文词和负样本拼接后的序列，长度统一为当前 batch 中最大长度  
- `masks`：掩码，标记哪些位置是真实词（1）或填充（0）  
- `labels`：标签，真实上下文词为 1，负样本和 padding 为 0  

请补全以下函数，使其返回四个 **PyTorch Tensor**，供 DataLoader 使用：

In [7]:
def batchify(data):
    """
    将一个批次的数据整理成训练所需的张量格式，包括：
    - centers: 中心词
    - contexts_negatives: 背景词和噪声词拼接并补齐
    - masks: 标记真实词和 padding
    - labels: 背景词为 1，噪声词和 padding 为 0

    参数：
    data : list 类型，每个元素是一个三元组 (center, context, negative)
           - center: 一个中心词索引
           - context: 背景词索引列表
           - negative: 噪声词索引列表
    """

    # TODO: 
    # 提示：
    # 1. 先计算每个 batch 中 context + negative 的最大长度 max_len
    # 2. 对每条样本：
    #      - 构造 centers 列表
    #      - 拼接 context + negative，并 pad 到 max_len，形成 contexts_negatives
    #      - 构造 mask：真实词为 1，padding 为 0
    #      - 构造 labels：context 为 1，negative 和 padding 为 0
    # 3. 最后返回四个 Tensor，类型为 torch.float（mask 和 labels 建议转换为 float）

In [ ]:
dataset = list(zip(all_centers, all_contexts, all_negatives))
batch_size = 512
data_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                       collate_fn=batchify)

## Part 6: 实现 Skip-gram 模型

在 Skip-gram 模型中，我们使用 **中心词嵌入** 和 **上下文/负样本嵌入** 计算预测分数。  
模型核心操作是通过 **批量矩阵乘法 (batch matrix multiplication)** 计算每个中心词和其上下文/负样本的相似度。

请补全以下函数，实现 Skip-gram 的前向计算：

In [8]:
def skip_gram(center, contexts_and_negatives, embed_v, embed_u):
    """
    参数：
    center: Tensor, (batch_size, 1)，中心词索引
    contexts_and_negatives: Tensor, (batch_size, num_context_negatives)，上下文词 + 负样本索引
    embed_v: nn.Embedding, 中心词嵌入层
    embed_u: nn.Embedding, 上下文/负样本嵌入层

    返回：
    pred: Tensor, (batch_size, 1, num_context_negatives)，预测分数
    """
    
    # TODO: 
    # 提示：
    # 1. 通过 embed_v(center) 获取中心词嵌入 v
    # 2. 通过 embed_u(contexts_and_negatives) 获取上下文/负样本嵌入 u
    # 3. 使用 torch.bmm 进行批量矩阵乘法计算预测分数
    # 4. 返回 pred，形状应为 (batch_size, 1, num_context_negatives)

## Part 7: 实现损失函数 (Binary Cross-Entropy Loss)

在 Skip-gram + 负采样中，我们使用 **Binary Cross-Entropy Loss** 来衡量中心词和上下文/负样本的匹配程度。  

### 公式
$$
\text{BCE}(p, y) = - [y \log(\sigma(p)) + (1-y) \log(1-\sigma(p))]
$$

- \(p\) : 模型预测的 raw logits  
- \(y\) : 标签 (context=1, negative/padding=0)  
- \(\sigma(p)\) : Sigmoid 函数  

为了屏蔽 padding 对损失的影响，需要使用 **mask**：
$$
\text{loss} = \frac{\sum (\text{BCE}(p, y) \cdot \text{mask})}{\sum \text{mask}}
$$

In [9]:
def sigmoid_binary_cross_entropy(pred, label, mask):
    """
    参数：
    pred: Tensor, 预测分数，形状 (batch_size, 1, num_context_negatives)
    label: Tensor, 标签，context=1, negative/padding=0，形状 (batch_size, num_context_negatives)
    mask: Tensor, 掩码，真实词=1, padding=0，形状 (batch_size, num_context_negatives)

    返回：
    loss: Tensor, batch 平均损失 (标量)
    """
    
    # TODO: 自行实现损失函数
    # 提示：
    # 1. 对 pred 应用 torch.sigmoid() 计算概率
    # 2. 按 Binary Cross-Entropy 公式计算每个位置的损失
    # 3. 乘上 mask，忽略 padding
    # 4. 对 batch 求平均返回标量 loss

## Part 8: 初始化词向量嵌入 (Embedding Layers)

在 Skip-gram 模型中，需要两个嵌入层：
- **中心词嵌入** `embed_v`：负责将中心词索引映射为向量  
- **上下文/负样本嵌入** `embed_u`：负责将上下文词和负样本索引映射为向量  

### 提示
- 词表大小：`vocab_size = len(idx_to_token)`  
- 词向量维度：`embed_size = 100`（可自行调整实验进行尝试）  
- 使用 PyTorch 的 **nn.Embedding** 层来创建嵌入  
- 两个嵌入层初始化方法类似，但是不同的对象
- Embedding 层会在训练过程中被更新，不需要自己手动初始化权重

In [ ]:
# TODO: 初始化 embed_v
embed_v = ...

# TODO: 初始化 embed_u
embed_u = ...

## Part 9: 实现训练循环

在本部分，你需要完成训练函数 `train` 中每个批次的核心计算逻辑。  

In [11]:
def train(embed_v, embed_u, lr, num_epochs, data_iter, device):
    embed_v.to(device)
    embed_u.to(device)
    optimizer = optim.Adam(list(embed_v.parameters()) + list(embed_u.parameters()), lr=lr)

    for epoch in range(num_epochs):
        l_sum, n = 0.0, 0
        start = time.time()
        for batch in data_iter:
            centers, contexts_negatives, mask, labels = [x.to(device) for x in batch]
            labels = labels.float()
            
            # TODO: 完成训练循环
            # 提示：
            # - 将批次数据输入 skip_gram 模型得到预测
            # - 使用 sigmoid_binary_cross_entropy 计算本次损失 l，注意 mask
            # - 进行反向传播和参数更新（使用 optimizer）


            # 累计本批次 loss 和样本数，用于输出平均 loss
            l_sum += l.item() * mask.size(0)
            n += mask.size(0)

        print(f'epoch {epoch+1}, loss {l_sum/n:.4f}, time {time.time()-start:.2f}s')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 你可以尝试使用不同的学习率 lr 和训练轮数 num_epochs，以达到更好的效果
train(embed_v, embed_u, lr=0.005, num_epochs=15, data_iter=data_iter, device=device)

## Part 10: 最近邻词测试（找同类词）

为了观察训练得到的词向量效果，我们已经提供了 `get_similar_tokens` 函数。  

请学生执行以下操作：

1. 使用提供的函数，输入不同的查询词（例如 `'computer'`、`'chip'` 等）  
2. 输出对应的前 k 个相似词  
3. 记录每次查询的结果，并思考以下问题：
   - 输出的词是否语义相关？  
   - 有哪些意外的相似词？为什么会出现？  
   - 模型训练过程中可能导致这些结果的原因

In [13]:
def get_similar_tokens(query_token, k, embed):
    W = embed.weight.data  # (vocab_size, embed_size)
    x = W[token_to_idx[query_token]]  # (embed_size,)

    cos = torch.matmul(W, x) / (
        torch.norm(W, dim=1) * torch.norm(x) + 1e-9
    )

    topk = torch.topk(cos, k=k+1).indices.tolist()

    print(f"Top {k} words similar to '{query_token}':")
    for i in topk[1:]:
        print(f"cosine sim={cos[i].item():.3f}: {idx_to_token[i]}")

In [ ]:
get_similar_tokens('computer', 3, embed_v) 

## 作业要求

请你补全以上所有 TODO 部分，完成以下任务：

1. 训练模型，并保留记录 loss 的输出记录（提交时保留notebook里的记录）。  
2. 输出 5 个 query token 的最近邻词结果，并进行简要分析，思考词向量是否捕捉到语义相似性。  
3. 如果你对超参数（包括词向量维度 `embed_size`，学习率 `lr` 和训练轮数 `num_epochs`）进行了探索，请思考并分析这些超参数对训练过程、loss 收敛情况以及最终词向量质量的影响。  

第2、3条要求可以在下面新建一个markdown单元格进行分析。